# XPS: VAMAS, Avantage and CasaXPS

`XPSData` holds one region of an XPS measurement: counts over `binding_energy`
in eV. A VAMAS file with several regions reads into a `dict`. The files are the
test fixtures under `tests/spectroscopy/xps/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lab_data_parser as ldp

In [ ]:
from lab_data_parser.spectroscopy.xps import DOUBLETS, doublet

XPS = Path("..") / "tests" / "spectroscopy" / "xps"
regions = ldp.XPSData.from_vamas(XPS / "vamas" / "sample-0001_3.vms")
{name: region.n_points for name, region in regions.items()}

## Survey

In [ ]:
survey = regions["XPS Survey"].to_cps()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(survey.binding_energy, survey.values)
ax.invert_xaxis()
ax.set_xlabel("binding energy / eV")
ax.set_ylabel("intensity / counts s^-1")
plt.show()

## Charge correction

Adventitious carbon is set to 284.8 eV. The measured position is read off the
C 1s maximum here; the shift is then applied to every region.

In [ ]:
c1s = regions["C1s Scan"].to_cps()
measured_eV = float(c1s.binding_energy[np.argmax(c1s.values)])
calibrated = {name: region.to_cps().calibrate_binding_energy(284.8, measured_eV=measured_eV)
              for name, region in regions.items()}
print(f"C 1s measured at {measured_eV:.2f} eV, shift {284.8 - measured_eV:+.2f} eV")

## Peak fit of Mo 3d

`fit_peaks` subtracts a Shirley background in the window and fits the peaks;
`doublet` builds both spin-orbit lines with the nominal splitting and area ratio.

In [ ]:
mo3d = calibrated["Mo3d Scan"]
peaks = doublet("Mo(VI) 3d", 232.6, fwhm_eV=1.2, lineshape="GL(30)", **DOUBLETS["Mo 3d"])
fitted = mo3d.fit_peaks(peaks, background="shirley", min_eV=228, max_eV=240)
fitted.peak_table()

In [ ]:
window = fitted.select_binding_energy_range(228, 240)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(window.binding_energy, window.values, "k.", ms=3, label="data")
ax.plot(window.binding_energy, window.envelope, label="envelope")
ax.plot(window.binding_energy, window.background, ls="--", label="background")
ax.invert_xaxis()
ax.set_xlabel("binding energy / eV")
ax.set_ylabel("intensity / counts s^-1")
ax.legend()
plt.show()

## Other readers

The Avantage `.avg` text dump of one region and the export of a CasaXPS fit.

In [ ]:
avg = ldp.XPSData.from_avantage_avg(XPS / "avantage" / "C1s Scan.avg")
print(avg.region, avg.n_points, avg.data_type)

casa = ldp.XPSData.from_casaxps_export(XPS / "casaxps_export_doublet.txt")
casa.peak_table()